# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available Record Sets (by their @id)
print('Record Sets in this dataset:')
for rs in dataset.record_sets:
    print(f"@id: {rs['@id']}, name: {rs.get('name', '<no_name>')}, description: {rs.get('description', '')}")

# Let's list for each RecordSet their Fields and Columns
for rs in dataset.record_sets:
    print(f"\nRecordSet: {rs['@id']} ({rs.get('name', '')}) fields and columns:")
    if 'field' in rs:
        for field in rs['field']:
            print(f"  Field @id: {field['@id']}, name: {field.get('name', '')}")
            if 'column' in field:
                for column in field['column']:
                    print(f"    Column @id: {column['@id']}, name: {column.get('name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this notebook, we will extract all tabular record sets.
# Usually, the relevant RecordSet for tabular data contains 'table' or 'matrix' in name or description.
# Let's collect all record set @ids.

record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
print('Available Record Sets @id:', record_sets_ids)

# Let's load all available record sets into DataFrames
dataframes = {}
for record_set_id in record_sets_ids:
    print(f"Loading RecordSet: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) == 0:
            print(f"  No records found for RecordSet: {record_set_id}")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records, columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"  Could not load RecordSet {record_set_id}: {e}")

# Let's pick the first loaded DataFrame as the focus for EDA
main_record_set_id = next(iter(dataframes)) if dataframes else None
if main_record_set_id:
    print(f"\nColumns in main DataFrame ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No tabular data available to load.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Identify numeric fields in the chosen RecordSet
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # We'll select the first numeric column
    numeric_fields = df.select_dtypes(include=np.number).columns.tolist()
    print(f"Numeric fields: {numeric_fields}")

    if numeric_fields:
        numeric_field = numeric_fields[0]
        threshold = df[numeric_field].mean() # or set a number like 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean value):")
        display(filtered_df.head())

        # Normalize numeric_field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a candidate categorical field
        # Heuristically, pick a non-numeric field with relatively low cardinality
        candidate_group_fields = [
            c for c in df.columns
            if c not in numeric_fields and df[c].nunique() < 20
        ]
        if candidate_group_fields:
            group_field = candidate_group_fields[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped mean for {numeric_field} by {group_field}:")
            display(grouped_df[[numeric_field]].head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field available for EDA analysis.")
else:
    print("No tabular data loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If we have a categorical field with low cardinality, draw boxplot
    if candidate_group_fields:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[candidate_group_fields[0]], y=df[numeric_field])
        plt.title(f'{numeric_field} by {candidate_group_fields[0]}')
        plt.xlabel(candidate_group_fields[0])
        plt.ylabel(numeric_field)
        plt.show()
else:
    print('No numeric fields to plot.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we:
- Loaded the Mass Spectrometry Analysis of Extracellular Vesicles dataset using its Croissant metadata.
- Inspected available record sets, fields, and their relationships using their Croissant `@id`s.
- Loaded tabular record sets into Pandas DataFrames and explored their columns.
- Performed basic exploratory data analysis on numeric columns, including filtering, normalization, and grouping.
- Visualized data distributions and possible relationships when categorical fields were available.

For more advanced analyses, consult the detailed schema and consider downstream applications relevant to mass spectrometry and extracellular vesicle protein profiles.